# Segmenting Credit-Card Customers with PCA and K-means

**Question.** 8,950 credit-card customers, no labels. Do they fall into distinct
behavioural groups, and how many?

**Data.** The *Credit Card Dataset for Clustering*, restricted to seven behavioural
features — how often a customer carries a balance, how often and in what form they
purchase, how often they take cash advances, how often they pay in full, and their tenure.

**Method.** K-means, evaluated by silhouette score across three feature representations:
raw, standardised, and PCA-reduced.

**Headline result.** Standardising changes the answer completely. On raw features the
silhouette looks excellent — but that number is an artefact of one feature's scale, not
evidence of good clustering. The section below makes that explicit, because it is the
single most important thing this notebook demonstrates.

## 1. Setup

Seeds are pinned throughout so every number here reproduces exactly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
sns.set_theme(style="whitegrid")

# The seven behavioural features, named rather than indexed by position.
FEATURES = [
    "BALANCE_FREQUENCY",                 # how often a balance is carried      (0-1)
    "PURCHASES_FREQUENCY",               # how often purchases are made        (0-1)
    "ONEOFF_PURCHASES_FREQUENCY",        # ... as one-off purchases            (0-1)
    "PURCHASES_INSTALLMENTS_FREQUENCY",  # ... as instalment purchases         (0-1)
    "CASH_ADVANCE_FREQUENCY",            # how often cash advances are taken   (0-1)
    "PRC_FULL_PAYMENT",                  # share of balance paid in full       (0-1)
    "TENURE",                            # months as a customer               (6-12)
]

## 2. Load and inspect missingness

In [ ]:
df = pd.read_csv("../data/CC.csv")
print(f"{df.shape[0]:,} customers x {df.shape[1]} columns")

missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False).to_frame("missing")

Two columns have gaps: `MINIMUM_PAYMENTS` (313) and `CREDIT_LIMIT` (1). Neither is among
the seven features used here, but they are imputed anyway so the frame is clean.

Imputing with the **median** rather than the mean — these are heavily right-skewed
monetary columns, where the mean is dragged by outliers.

In [ ]:
numeric = df.select_dtypes(include=np.number).columns
df[numeric] = df[numeric].fillna(df[numeric].median())

assert df[numeric].isnull().sum().sum() == 0, "missing values remain"

X = df[FEATURES].copy()
X.describe().T[["mean", "std", "min", "max"]].round(3)

### The scale problem, in one table

Six of the seven features are bounded in **[0, 1]**. `TENURE` runs **6 to 12** — an order
of magnitude wider. K-means minimises squared Euclidean distance, so a feature with a
larger numeric range contributes disproportionately to every distance computation.

In effect, clustering these features unscaled means *clustering almost entirely on
tenure*. That matters for how the next result is read.

## 3. How many clusters? (elbow method)

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=FEATURES)

wcss = []
ks = range(1, 11)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, wcss, marker="o")
ax.set_xlabel("number of clusters (k)")
ax.set_ylabel("within-cluster sum of squares")
ax.set_title("Elbow method (standardised features)")
ax.set_xticks(list(ks))
fig.tight_layout()
fig.savefig("../results/elbow_scaled.png", dpi=120)
plt.show()

The curve bends between **k = 3 and k = 4** without a single decisive elbow — common on
behavioural data, where groups shade into each other rather than separating cleanly.
Silhouette score is used below to choose between them on a firmer basis.

## 4. Does scaling matter? Comparing three representations

The key methodological point: a silhouette score is **only comparable within the feature
space the model was fit in.** Scoring clusters fitted on scaled data against the *unscaled*
matrix measures nothing meaningful. Each row below fits and scores in the same space.

In [ ]:
pca_full = PCA(n_components=2, random_state=SEED)
X_pca = pca_full.fit_transform(X_scaled)

spaces = {
    "raw features":        X.to_numpy(),
    "standardised":        X_scaled.to_numpy(),
    "PCA(2) of scaled":    X_pca,
}

rows = []
for name, matrix in spaces.items():
    for k in (2, 3, 4):
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
        labels = km.fit_predict(matrix)
        # scored in the SAME space the model was fitted in
        rows.append({"space": name, "k": k,
                     "silhouette": silhouette_score(matrix, labels)})

results = pd.DataFrame(rows).pivot(index="space", columns="k", values="silhouette")
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
results.T.plot(marker="o", ax=ax)
ax.set_xlabel("number of clusters (k)")
ax.set_ylabel("silhouette score")
ax.set_title("Silhouette by feature representation")
ax.set_xticks([2, 3, 4])
ax.legend(title="")
fig.tight_layout()
fig.savefig("../results/silhouette_comparison.png", dpi=120)
plt.show()

### Why the raw score is a trap

Raw features produce by far the highest silhouette — and it is **not** a better clustering.

`TENURE` dominates the distance metric, and it is a coarse, highly concentrated variable
(most customers sit at 12 months). K-means therefore splits customers into a few tight
tenure bands. Tight, well-separated bands are exactly what silhouette rewards, so the
score is high while the segmentation is nearly useless: it recovers a variable already
known, and ignores the purchasing behaviour actually of interest.

Standardising removes that dominance and lets all seven features contribute. The score
falls — and the clustering gets more informative. **A higher silhouette is not
automatically a better model.**

## 5. What PCA retains

In [ ]:
pca_check = PCA(random_state=SEED).fit(X_scaled)
evr = pca_check.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(evr) + 1), evr, color="steelblue", edgecolor="black", label="individual")
ax.plot(range(1, len(evr) + 1), np.cumsum(evr), marker="o", color="crimson", label="cumulative")
ax.set_xlabel("principal component")
ax.set_ylabel("explained variance ratio")
ax.set_title("PCA explained variance")
ax.legend()
fig.tight_layout()
fig.savefig("../results/pca_explained_variance.png", dpi=120)
plt.show()

print(f"PC1 + PC2 retain {evr[:2].sum():.1%} of total variance")

## 6. The chosen segmentation

In [ ]:
BEST_K = int(results.loc["standardised"].idxmax())
print(f"best k on standardised features: {BEST_K}")

km = KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED)
labels = km.fit_predict(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="viridis", s=8, alpha=0.6)
ax.set_xlabel(f"PC1 ({evr[0]:.1%} of variance)")
ax.set_ylabel(f"PC2 ({evr[1]:.1%} of variance)")
ax.set_title(f"Customer segments (k={BEST_K}), projected onto first two components")
ax.legend(*sc.legend_elements(), title="cluster")
fig.tight_layout()
fig.savefig("../results/cluster_scatter_pca.png", dpi=120)
plt.show()

### Reading the segments

Cluster centres, returned to original units, describe what each group actually does.

In [ ]:
centres = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_), columns=FEATURES
)
centres.index.name = "cluster"
profile = centres.round(2)
profile["n customers"] = pd.Series(labels).value_counts().sort_index()
profile

## Findings

1. **Scaling is the decisive preprocessing step.** Unscaled, `TENURE` (range 6–12)
   overwhelms six features bounded in [0, 1], so K-means effectively clusters on tenure
   alone. The silhouette score *rewards* this, which is why the raw number looks best and
   means least.
2. **Silhouette is only comparable within a fixed feature space.** Fitting on scaled data
   and scoring against the unscaled matrix — an easy mistake — produces a number that
   describes neither model.
3. **PCA to two components is for visualisation here, not performance.** It compresses
   seven features into a plottable plane at a real cost in retained variance; the
   comparison table shows what that costs.
4. **The elbow is genuinely ambiguous** between k = 3 and k = 4. Real behavioural data
   rarely separates cleanly, and reporting the ambiguity is more honest than picking the
   prettier plot.

## Limitations

- Silhouette measures geometric separation, not business usefulness. A well-separated
  segment is not necessarily an actionable one.
- K-means assumes roughly spherical, similarly sized clusters. Several of these features
  are strongly skewed and zero-inflated, which violates that assumption; DBSCAN or a
  Gaussian mixture would be a fairer comparison.
- Only 7 of 17 available columns are used, inherited from the original analysis. The
  monetary columns (`BALANCE`, `PURCHASES`, `CREDIT_LIMIT`) are excluded and would likely
  carry real segmentation signal.
- Median imputation on `MINIMUM_PAYMENTS` (313 missing) assumes the values are missing at
  random, which is untested.
- No external validation — there is no ground-truth segmentation to check against.